# Mobile Phone Retail and Sales Management Database
## Data generation, relational design, validation and analysis

This notebook documents the construction of a fictional mobile-phone retailer database. All records are synthetic and are intended only for database practice. Customer email addresses use `example.org`; no real contact, payment-card or IMEI information is included.

**Reproducibility:** run the cells from top to bottom in the same folder as `generate_database.py`. The generator uses a fixed random seed (20261106), so it recreates the same data-quality cases and row counts.

In [ ]:
from pathlib import Path
import sqlite3
from generate_database import create_database

db_path = create_database(Path("mobile_phone_retail.sqlite"))
print("Database created:", db_path.resolve())


## 1. Relational schema and design rationale

The database contains six related tables:

- `customers`: one row per fictional customer.
- `products`: one row per phone variant, including brand, model, storage, colour, operating system, screen size, launch year, price and condition.
- `stores`: physical branches and an online fulfilment location.
- `orders`: transaction header, linked to one customer and one store.
- `order_items`: products and quantities within each order.
- `store_inventory`: stock of each product at each store.

`order_items` resolves the many-to-many relationship between orders and products. `store_inventory` resolves the many-to-many relationship between stores and products. Both use composite primary keys: `(order_id, product_id)` and `(store_id, product_id)`. The product table also has a composite `UNIQUE` constraint to prevent repeated product variants.

In [ ]:
con = sqlite3.connect(db_path)
con.execute("PRAGMA foreign_keys=ON")

for table in ["customers", "products", "stores", "orders", "order_items", "store_inventory"]:
    count = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table:18s} {count:>6,} rows")

print("Foreign-key violations:", con.execute("PRAGMA foreign_key_check").fetchall())
con.close()


## 2. Data types and measurement scales

The database uses SQLite `TEXT`, `INTEGER` and `REAL` storage types. ISO-formatted dates are stored as `TEXT` so they remain readable and sortable.

| Measurement scale | Examples |
|---|---|
| Nominal | brand, colour, operating system, region, channel, payment method |
| Ordinal | age band, product condition, order status |
| Interval | launch year, customer-since year |
| Ratio | price, screen size, storage, quantity, discount, delivery days, stock quantity |

The values are generated within plausible ranges. Product prices vary by brand, storage capacity and condition; order dates are spread across 2026; and stock levels vary by store and product.

## 3. Deliberately inserted data-quality cases

Two data-quality issues are deliberately generated rather than left to chance:

1. **Missing values:** a small proportion of orders have `NULL` in `orders.delivery_days`. A delivery estimate may not yet be available when an order is entered. The fixed-seed run creates **59 missing estimates**.
2. **Duplicate business events:** **35 repeated business-event groups** are inserted using new order IDs. The customer, date, product, quantity and unit price are repeated, while surrogate primary keys remain unique. This simulates a transaction imported more than once and demonstrates why checking only primary keys is insufficient.

The following excerpts are read directly from the generator so the deliberate insertion logic is visible alongside the database-generation process.

In [ ]:
source = Path("generate_database.py").read_text(encoding="utf-8").splitlines()

def show_block(start_text, end_text):
    start = next(i for i, line in enumerate(source) if start_text in line)
    end = next((i for i in range(start + 1, len(source)) if end_text in source[i]), len(source))
    print("\n".join(f"{i+1:03d}: {source[i]}" for i in range(start, end)))

print("MISSING-VALUE INSERTION")
show_block("# Deliberately inject missing delivery estimates", "order_rows.append")

print("\nDUPLICATE BUSINESS-EVENT INSERTION")
show_block("# Deliberately create duplicate business events", "con.commit()")


In [ ]:
con = sqlite3.connect(db_path)
missing = con.execute(
    "SELECT COUNT(*) FROM orders WHERE delivery_days IS NULL"
).fetchone()[0]

duplicate_groups = con.execute('''
    SELECT COUNT(*) FROM (
        SELECT o.customer_id, o.order_date, i.product_id,
               i.quantity, i.unit_price_gbp
        FROM orders o JOIN order_items i USING(order_id)
        GROUP BY o.customer_id, o.order_date, i.product_id,
                 i.quantity, i.unit_price_gbp
        HAVING COUNT(*) > 1
    )
''').fetchone()[0]

print("Missing delivery estimates:", missing)
print("Duplicate business-event groups:", duplicate_groups)
print("Foreign-key violations:", con.execute("PRAGMA foreign_key_check").fetchall())
con.close()


## 4. Example business analysis

The SQL file contains queries for revenue by brand, monthly sales, popular models, customer age bands, payment methods, store performance and low-stock alerts. It also includes missing-value and duplicate-event audits.

Example: revenue by brand is calculated from completed order lines using the transaction price and discount. The analysis is descriptive of this generated dataset only; it must not be presented as evidence of real UK retail behaviour.

In [ ]:
con = sqlite3.connect(db_path)
query = '''
SELECT p.brand,
       ROUND(SUM(i.quantity * i.unit_price_gbp *
                 (1 - i.discount_pct / 100.0)), 2) AS revenue_gbp
FROM order_items i
JOIN products p USING(product_id)
JOIN orders o USING(order_id)
WHERE o.order_status = 'Completed'
GROUP BY p.brand
ORDER BY revenue_gbp DESC
'''
for row in con.execute(query):
    print(row)
con.close()


## 5. Ethics, privacy and limitations

All records are synthetic. A real retailer should identify a lawful basis for processing, explain its use of personal data, collect only what is necessary, maintain accuracy, apply access controls and security, set retention periods and support data-subject rights under the UK GDPR and Data Protection Act 2018. Marketing consent should be kept separate from information needed to fulfil an order.

The dataset cannot establish actual customer preferences or UK market trends. A production system would also require stronger access management, audit logging, backups and additional entities such as suppliers, returns and stock movements.